# Ingestão e análise - Antecipação de Recebíveis da Olist

O objetivo desse mini-projeto é aplicar os conceitos de **Engenharia e Análise de Dados** para estudo e construção de portfólio.\
Abaixo um pipeline de dados (camadas **bronze → silver → gold**) foi montado completamente usando os recursos do Databricks com scripts em **Python, PySpark e SQL**.\
\
**Dataset utilizado**: Kaggle -  [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

## 1. Bronze - ingestão dos CSVs

Lê os CSVs da Olist no volume raw e grava uma tabela Delta por arquivo no schema `workspace.olist`.

**Regra da camada bronze:** nenhuma transformação de negócio. Todas as colunas entram como string, exatamente como vieram, mais uma coluna de linhagem (`_ingestao_em`). Tipos e limpeza ficam na camada silver.

In [0]:
# Importando as bibliotecas necessárias
from pyspark.sql import functions as F, Window

# Path dos arquivos originais baixados do Kaggle
base = "/Volumes/workspace/olist/raw"

# Função para carregar os arquivos originais e gravar na camada bronze
def bronze(arquivo, tabela):
    (spark.read.option("header", True).csv(f"{base}/{arquivo}")
        .withColumn("_ingestao_em", F.current_timestamp())
        .write.mode("overwrite").saveAsTable(f"workspace.olist.{tabela}"))

bronze("olist_orders_dataset.csv", "bronze_orders")
bronze("olist_customers_dataset.csv", "bronze_customers")
bronze("olist_order_payments_dataset.csv", "bronze_payments")
bronze("olist_order_items_dataset.csv", "bronze_items")
bronze("olist_products_dataset.csv", "bronze_products")
bronze("olist_sellers_dataset.csv", "bronze_sellers")

## 2. Silver - tratamento e rateio por vendedor (Etapa 1)

Duas partes:

1. **Tabelas tipadas**: cada tabela da bronze ganha tipos corretos.
2. **`fct_venda_seller`**: uma linha por pedido × vendedor × pagamento, com o valor do pagamento rateado entre os vendedores do pedido. É a base de todas as etapas seguintes.

#### 2.1. Tabelas tipadas

In [0]:
# Dataframe de pedidos, formatando datas e criando coluna flag_cancelado
orders_silver = (spark.table("workspace.olist.bronze_orders")
    .select(
        "order_id",
        "customer_id",
        "order_status",
        F.col("order_purchase_timestamp").cast("timestamp").alias("order_purchase_ts"),
        F.col("order_approved_at").cast("timestamp").alias("order_approved_ts"),
        F.col("order_delivered_carrier_date").cast("timestamp").alias("order_delivered_carrier_ts"),
        F.col("order_delivered_customer_date").cast("timestamp").alias("order_delivered_customer_ts"),
        F.col("order_estimated_delivery_date").cast("timestamp").alias("order_estimated_delivery_ts"),
        (F.col("order_status") == "canceled").alias("flag_cancelado")))

orders_silver.write.mode("overwrite").saveAsTable("workspace.olist.silver_orders")


In [0]:
# Dataframe de itens
items_silver = (spark.table("workspace.olist.bronze_items")
    .select(
        "order_id",
        F.col("order_item_id").cast("int").alias("order_item_id"),
        "product_id",
        "seller_id",
        F.col("shipping_limit_date").cast("timestamp").alias("shipping_limit_ts"),
        F.col("price").cast("double").alias("price"),
        F.col("freight_value").cast("double").alias("freight_value")))

items_silver.write.mode("overwrite").saveAsTable("workspace.olist.silver_items")

In [0]:
# Dataframe de vendedores
sellers_silver = (spark.table("workspace.olist.bronze_sellers")
    .select(
        "seller_id",
        F.col("seller_zip_code_prefix").alias("cep_prefixo"),
        F.col("seller_city").alias("cidade"),
        F.upper(F.col("seller_state")).alias("uf")))

sellers_silver.write.mode("overwrite").saveAsTable("workspace.olist.silver_sellers")

**Pagamentos**: há casos de cartão de crédito com `payment_installments = 0`, o que é erro de cadastro (toda venda no crédito tem ao menos 1 parcela). A coluna `parcelas` corrige para 1 e `flag_parcela_corrigida` guarda o registro da correção. A coluna original fica preservada.

In [0]:
# Dataframe de pagamentos
items_silver = spark.sql("""
WITH tipado AS (
  SELECT
    order_id,
    try_cast(payment_sequential AS INT) AS payment_sequential,
    payment_type,
    try_cast(payment_installments AS INT) AS parcelas_original,
    try_cast(payment_value AS DOUBLE) AS payment_value
  FROM workspace.olist.bronze_payments
)
SELECT
  *,
  CASE
    WHEN payment_type = 'credit_card' AND coalesce(parcelas_original, 0) < 1 THEN 1
    WHEN payment_type <> 'credit_card' THEN 1
    ELSE parcelas_original
  END AS parcelas,
  payment_type = 'credit_card' AND coalesce(parcelas_original, 0) < 1 AS flag_parcela_corrigida
FROM tipado
""")

items_silver.write.mode("overwrite").saveAsTable("workspace.olist.silver_order_payments")

**Produtos**: para efeito dessa análise, a categoria funciona como uma simulação de MCC (ramo de atividade do lojista). Feito um coalesce para tratar produtos sem cateogoria.

In [0]:
# Dataframe de produtos
products_silver = spark.sql("""
SELECT
  product_id,
  COALESCE(product_category_name, 'sem_categoria') AS categoria
FROM workspace.olist.bronze_products
""")

products_silver.write.mode("overwrite").saveAsTable("workspace.olist.silver_products")

#### 2.2. `f_venda_seller` - rateio do pagamento entre vendedores

O pagamento vem por pedido, mas a análise é por lojista. Quando um pedido tem itens de mais de um vendedor, o pagamento é dividido pela participação de cada um no valor do pedido:

`pct_seller = (price + freight do vendedor) ÷ (price + freight do pedido)`

Outras regras de negócio aplicadas:

- **Data da venda (D0)** = `order_approved_ts`. Pedidos nunca aprovados ficam de fora: não geram recebível.
- **Categoria do vendedor no pedido** = a do item de maior preço (`max_by`).
- **`meio_pagamento`** traduz o `payment_type` para a linguagem de adquirência: débito, crédito à vista, crédito parcelado, boleto, voucher.
- Pedidos cancelados ficam na tabela com `flag_cancelado = true`. Quem decide filtrar é a etapa seguinte.

In [0]:
f_venda_seller = spark.sql("""
 WITH itens AS (
   SELECT
     i.order_id,
     i.seller_id,
     sum(i.price + i.freight_value) AS valor_itens_seller,
     count(*) AS qtd_itens,
     max_by(p.categoria, i.price) AS categoria
   FROM workspace.olist.silver_items i
   LEFT JOIN workspace.olist.silver_products p ON p.product_id = i.product_id
   GROUP BY i.order_id, i.seller_id
 ),
 share AS (
   SELECT
     *,
     valor_itens_seller / sum(valor_itens_seller) OVER (PARTITION BY order_id) AS pct_seller,
     count(*) OVER (PARTITION BY order_id) AS qtd_sellers_pedido
   FROM itens
 )
 SELECT
   o.order_id,
   s.seller_id,
   s.categoria,
   CAST(o.order_approved_ts AS DATE) AS data_venda,
   p.payment_sequential,
   p.payment_type,
   CASE
     WHEN p.payment_type = 'credit_card' AND p.parcelas > 1 THEN 'credito_parcelado'
     WHEN p.payment_type = 'credit_card' THEN 'credito_avista'
     WHEN p.payment_type = 'debit_card' THEN 'debito'
     WHEN p.payment_type = 'boleto' THEN 'boleto'
     WHEN p.payment_type = 'voucher' THEN 'voucher'
     ELSE 'outros'
   END AS meio_pagamento,
   p.parcelas,
   p.payment_value * s.pct_seller AS valor_venda,
   s.pct_seller,
   s.qtd_sellers_pedido,
   o.flag_cancelado,
   p.flag_parcela_corrigida
 FROM workspace.olist.silver_orders o
 JOIN workspace.olist.silver_order_payments p ON p.order_id = o.order_id
 JOIN share s ON s.order_id = o.order_id
 WHERE o.order_approved_ts IS NOT NULL
   AND s.valor_itens_seller > 0
""")

f_venda_seller.write.mode("overwrite").saveAsTable("workspace.olist.silver_f_venda_seller")